In [1]:
import re
import sys
import yaml
from pathlib import Path
from seeq import spy

_ROOT   = Path.cwd().parent
_NB_DIR = Path.cwd()

print(f"Projeto : {_ROOT.name}")
print(f"Usuário : {spy.user.username}")

Projeto : datalab
Usuário : 00urj21b6w22YCAhB1t7


## Passo 1 — URL da worksheet

Abra no Seeq a worksheet que contém os sinais de **Força A**, **Força B** e **Phantom**.
Copie o URL do navegador e cole na variável abaixo.

In [2]:
WORKSHEET_URL = "https://kcc.seeq.site/workbook/0F14732A-4A28-FFD0-82B4-0AFD16024E6D/worksheet/0F148C10-61CB-EA30-80F7-6A20F43D0430"

## Passo 2 — Classificação automática dos sinais

O sistema separa automaticamente os sinais pelo nome:
- Nomes com `phantom`, `product`, `code`, `mes`, `bagger` → sinal de **Phantom**
- Demais sinais → sinais de **Força** (Forca_A e Forca_B por ordem de exibição)

Execute a célula abaixo para ver a classificação.

In [3]:
_PHANTOM_KW = {"mes", "phantom", "code", "sku", "product", "produto", "bagger"}

m = re.search(r'/workbook/([^/?#\s]+)/worksheet/([^/?#\s]+)', WORKSHEET_URL)
if not m:
    raise ValueError(
        "URL não reconhecida. Formato esperado:\n"
        "  https://kcc.seeq.site/workbook/WORKBOOK-ID/worksheet/WORKSHEET-ID"
    )

workbook_id  = m.group(1)
worksheet_id = m.group(2)

print("Puxando sinais via spy.workbooks.pull() ...")
wbs = spy.workbooks.pull(workbook_id, include_referenced_workbooks=False, quiet=True)
ws  = next(
    (s for s in wbs[0].worksheets if s.id.upper() == worksheet_id.upper()),
    None,
)
if ws is None:
    raise ValueError(f"Worksheet '{worksheet_id}' não encontrada no workbook.")

print(f"Worksheet: {ws.name!r}\n")

force_rows, phantom_rows = [], []
for _, row in ws.display_items.iterrows():
    name_lower = str(row.get("Name", "")).lower()
    if any(kw in name_lower for kw in _PHANTOM_KW):
        phantom_rows.append(row)
    else:
        force_rows.append(row)

print(f"Força    ({len(force_rows)} sinais):")
for i, r in enumerate(force_rows):
    print(f"  [{i}] {r['Name']}")
print(f"\nPhantom  ({len(phantom_rows)} sinais):")
for i, r in enumerate(phantom_rows):
    print(f"  [{i}] {r['Name']}")

Puxando sinais via spy.workbooks.pull() ...
Worksheet: 'FB17'

Força    (2 sinais):
  [0] BRSZ FB17 Quality - FB17_8102_A1_Força Sel Frontal
  [1] BRSZ FB17 Quality - FB17_8102_A1_Força Sel Frontal

Phantom  (1 sinais):
  [0] BRSZ_FB17_MES_Phantom_Code


## Passo 3 — Confirmar seleção

Verifique os índices na lista acima.
Se a classificação estiver correta, mantenha os valores padrão (`0` e `1`).
Se algum sinal estiver errado, ajuste os índices abaixo antes de executar.

In [4]:
# Ajuste os índices se necessário
IDX_FORCA_A = 0
IDX_FORCA_B = 1
IDX_PHANTOM = 0

# ── Validação ────────────────────────────────────────────────────────────────
if len(force_rows) < 2:
    raise ValueError(
        f"Preciso de ≥ 2 sinais de força, encontrei {len(force_rows)}.\n"
        "Verifique se a URL é da worksheet correta."
    )

r_a = force_rows[IDX_FORCA_A]
r_b = force_rows[IDX_FORCA_B]
r_p = phantom_rows[IDX_PHANTOM] if phantom_rows else None

print("Seleção confirmada:")
print(f"  Forca_A : {r_a['Name']}")
print(f"  Forca_B : {r_b['Name']}")
if r_p is not None:
    print(f"  Phantom : {r_p['Name']}")
else:
    print("  Phantom : ⚠  não detectado — normalização por produto desativada")

Seleção confirmada:
  Forca_A : BRSZ FB17 Quality - FB17_8102_A1_Força Sel Frontal
  Forca_B : BRSZ FB17 Quality - FB17_8102_A1_Força Sel Frontal
  Phantom : BRSZ_FB17_MES_Phantom_Code


## Passo 4 — Parâmetros

Ajuste se necessário (os padrões funcionam para a maioria dos casos).

In [5]:
# Quantos dias de histórico puxar do Seeq
TIME_DELTA_DAYS = 1460   # 4 anos

# Filtro IQR para leituras anômalas (1.0 = agressivo, 2.0 = conservador)
IQR_MULTIPLIER = 1.0

## Passo 5 — Gerar config.yaml

Execute a célula abaixo para criar o arquivo de configuração do projeto.

In [6]:
_DEFAULT_TRIGGER = {
    "weibull_beta":          1.181,
    "weibull_eta_h":         1297.0,
    "boost_sinal":           0.65,
    "vida_decay_w":          0.8,
    "limiar_p_risk":         0.48,
    "limiar_signal_score":   0.22,
    "idade_minima_dias":     15,
    "proj_48h_limiar":       800.0,
    "sustentacao_proj_dias": 2,
    "cooldown_h":            48,
    "snooze_dias":           5,
    "aviso_p_risk":          0.35,
    "aviso_signal":          0.15,
    "aviso_cooldown_h":      72,
    "risco_forca_limiar":    800.0,
    "risco_mediana_ok":      950.0,
    "risco_n_max":           1,
    "risco_cooldown_h":      48,
    "critico_forca_min":     800.0,
    "critico_p_risk_min":    0.30,
    "critico_cooldown_h":    48,
    "emergencial_min_eventos": 3,
    "emergencial_idade_frac":  0.85,
    "emergencial_p_risk_min":  0.40,
    "emergencial_cooldown_h":  48,
    "revisao_marcos_dias":   [20, 25, 35],
}

config = {
    "project": {
        "workbook_id":     workbook_id,
        "worksheet_name":  ws.name,
        "maquina":         ws.name,
        "time_delta_days": TIME_DELTA_DAYS,
    },
    "signals": [
        {"id": r_a["ID"], "name": "Forca_A", "original_name": r_a["Name"], "type": "Signal"},
        {"id": r_b["ID"], "name": "Forca_B", "original_name": r_b["Name"], "type": "Signal"},
    ],
    "phantom_signals": [
        {"id": r_p["ID"], "name": "phantom", "original_name": r_p["Name"], "type": "Signal"},
    ] if r_p is not None else [],
    "preprocessing": {
        "delta_col_a":    "Forca_A",
        "delta_col_b":    "Forca_B",
        "iqr_multiplier": IQR_MULTIPLIER,
    },
    "trigger": _DEFAULT_TRIGGER,
}

config_path = _ROOT / "config.yaml"
with open(config_path, "w", encoding="utf-8") as f:
    yaml.dump(config, f, allow_unicode=True, default_flow_style=False, sort_keys=False)

print(f"✓ {config_path} gerado.")
print(f"  Forca_A : {r_a['Name']}")
print(f"  Forca_B : {r_b['Name']}")
if r_p is not None:
    print(f"  Phantom : {r_p['Name']}")
print(f"  Máquina : {ws.name}")
print(f"  Histórico: {TIME_DELTA_DAYS} dias")
print("\nPróximos passos:")
print("  1. Gerar troca_modulo.csv  →  python extrair_troca_modulo.py --iw38 iw38.csv")
print("  2. Executar pipeline_producao.ipynb")

✓ /home/datalab/config.yaml gerado.
  Forca_A : BRSZ FB17 Quality - FB17_8102_A1_Força Sel Frontal
  Forca_B : BRSZ FB17 Quality - FB17_8102_A1_Força Sel Frontal
  Phantom : BRSZ_FB17_MES_Phantom_Code
  Máquina : FB17
  Histórico: 1460 dias

Próximos passos:
  1. Gerar troca_modulo.csv  →  python extrair_troca_modulo.py --iw38 iw38.csv
  2. Executar pipeline_producao.ipynb
